In [1]:
import sys
sys.path.append("..")
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import Compose

from da2_metric_depth.depth_anything_v2.util.transform import Resize, NormalizeImage, PrepareForNet
from da2_metric_depth.depth_anything_v2.dinov2 import DINOv2
from da2_metric_depth.depth_anything_v2.dpt import DPTHead
from da2_metric_depth.depth_anything_v2.util.blocks import FeatureFusionBlock, _make_scratch


def _make_fusion_block(features, use_bn, size=None):
    return FeatureFusionBlock(
        features,
        nn.ReLU(False),
        deconv=False,
        bn=use_bn,
        expand=False,
        align_corners=True,
        size=size,
    )


class ConvBlock(nn.Module):
    def __init__(self, in_feature, out_feature):
        super().__init__()
        
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_feature, out_feature, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_feature),
            nn.ReLU(True)
        )
    
    def forward(self, x):
        return self.conv_block(x)


class DPTHead(nn.Module):
    def __init__(
        self, 
        in_channels, 
        features=256, 
        use_bn=False, 
        out_channels=[256, 512, 1024, 1024], 
        use_clstoken=False
    ):
        super(DPTHead, self).__init__()
        
        self.use_clstoken = use_clstoken
        
        self.projects = nn.ModuleList([
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channel,
                kernel_size=1,
                stride=1,
                padding=0,
            ) for out_channel in out_channels
        ])
        
        self.resize_layers = nn.ModuleList([
            nn.ConvTranspose2d(
                in_channels=out_channels[0],
                out_channels=out_channels[0],
                kernel_size=4,
                stride=4,
                padding=0),
            nn.ConvTranspose2d(
                in_channels=out_channels[1],
                out_channels=out_channels[1],
                kernel_size=2,
                stride=2,
                padding=0),
            nn.Identity(),
            nn.Conv2d(
                in_channels=out_channels[3],
                out_channels=out_channels[3],
                kernel_size=3,
                stride=2,
                padding=1)
        ])
        
        if use_clstoken:
            self.readout_projects = nn.ModuleList()
            for _ in range(len(self.projects)):
                self.readout_projects.append(
                    nn.Sequential(
                        nn.Linear(2 * in_channels, in_channels),
                        nn.GELU()))
        
        self.scratch = _make_scratch(
            out_channels,
            features,
            groups=1,
            expand=False,
        )
        
        self.scratch.stem_transpose = None
        
        self.scratch.refinenet1 = _make_fusion_block(features, use_bn)
        self.scratch.refinenet2 = _make_fusion_block(features, use_bn)
        self.scratch.refinenet3 = _make_fusion_block(features, use_bn)
        self.scratch.refinenet4 = _make_fusion_block(features, use_bn)
        
        head_features_1 = features
        head_features_2 = 32
        
        self.scratch.output_conv1 = nn.Conv2d(head_features_1, head_features_1 // 2, kernel_size=3, stride=1, padding=1)
        self.scratch.output_conv2 = nn.Sequential(
            nn.Conv2d(head_features_1 // 2, head_features_2, kernel_size=3, stride=1, padding=1),
            nn.ReLU(True),
            nn.Conv2d(head_features_2, 1, kernel_size=1, stride=1, padding=0),
            # nn.ReLU(True),
            nn.Sigmoid(),
            nn.Identity(),
        )
    
    def forward(self, out_features, patch_h, patch_w):
        out = []
        for i, x in enumerate(out_features):
            if self.use_clstoken:
                x, cls_token = x[0], x[1]
                readout = cls_token.unsqueeze(1).expand_as(x)
                x = self.readout_projects[i](torch.cat((x, readout), -1))
            else:
                x = x[0]
            
            x = x.permute(0, 2, 1).reshape((x.shape[0], x.shape[-1], patch_h, patch_w))
            
            x = self.projects[i](x)
            x = self.resize_layers[i](x)
            
            out.append(x)
        
        layer_1, layer_2, layer_3, layer_4 = out
        
        layer_1_rn = self.scratch.layer1_rn(layer_1)
        layer_2_rn = self.scratch.layer2_rn(layer_2)
        layer_3_rn = self.scratch.layer3_rn(layer_3)
        layer_4_rn = self.scratch.layer4_rn(layer_4)
        
        path_4 = self.scratch.refinenet4(layer_4_rn, size=layer_3_rn.shape[2:])        
        path_3 = self.scratch.refinenet3(path_4, layer_3_rn, size=layer_2_rn.shape[2:])
        path_2 = self.scratch.refinenet2(path_3, layer_2_rn, size=layer_1_rn.shape[2:])
        path_1 = self.scratch.refinenet1(path_2, layer_1_rn)
        
        out = self.scratch.output_conv1(path_1)
        out = F.interpolate(out, (int(patch_h * 14), int(patch_w * 14)), mode="bilinear", align_corners=True)
        out = self.scratch.output_conv2(out)
        
        return out


class DepthAnythingV2(nn.Module):
    def __init__(
        self,
        encoder="vitl",
        features=256,
        out_channels=[256, 512, 1024, 1024],
        use_bn=False,
        use_clstoken=False,
        max_depth=20.0,
    ):
        super(DepthAnythingV2, self).__init__()

        self.intermediate_layer_idx = {
            "vits": [2, 5, 8, 11],
            "vitb": [2, 5, 8, 11],
            "vitl": [4, 11, 17, 23],
            "vitg": [9, 19, 29, 39],
        }

        self.max_depth = max_depth

        self.encoder = encoder
        self.pretrained = DINOv2(model_name=encoder)

        self.depth_head = DPTHead(
            self.pretrained.embed_dim,
            features,
            use_bn,
            out_channels=out_channels,
            use_clstoken=use_clstoken,
        )

    def forward(self, x):
        print(x.shape)
        patch_h, patch_w = x.shape[-2] // 14, x.shape[-1] // 14
        patch_grid_size = (patch_w, patch_h)
        print(patch_grid_size)
        features = self.pretrained.get_intermediate_layers(
            x, self.intermediate_layer_idx[self.encoder], return_class_token=True
        )

        fea_maps = []
        for i, fea in enumerate(features):
            fea_maps.append(fea[0])
            
        depth = self.depth_head(features, patch_h, patch_w) * self.max_depth
        # return depth.squeeze(1), var_maps
        return depth.squeeze(1), fea_maps
    
    @torch.no_grad()
    def infer_image(self, raw_image, input_size=518):
        image, (h, w) = self.image2tensor(raw_image, input_size)

        depth, var_maps = self.forward(image)

        depth = F.interpolate(
            depth[:, None], (h, w), mode="bilinear", align_corners=True
        )[0, 0]

        return depth.cpu().numpy(), var_maps

    def image2tensor(self, raw_image, input_size=518):
        transform = Compose(
            [
                Resize(
                    width=input_size,
                    height=input_size,
                    resize_target=False,
                    keep_aspect_ratio=True,
                    ensure_multiple_of=14,
                    resize_method="lower_bound",
                    image_interpolation_method=cv2.INTER_CUBIC,
                ),
                NormalizeImage(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                PrepareForNet(),
            ]
        )

        h, w = raw_image.shape[:2]

        image = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB) / 255.0

        image = transform({"image": image})["image"]
        image = torch.from_numpy(image).unsqueeze(0)

        DEVICE = (
            "cuda"
            if torch.cuda.is_available()
            else "mps" if torch.backends.mps.is_available() else "cpu"
        )
        image = image.to(DEVICE)

        return image, (h, w)

In [2]:
def get_variance_maps(features, use_cls_token=True):
    feature_maps = []
    if use_cls_token:
        for feature in features:
            fea = feature[0]
            feature_maps.append(fea)
    else:
        feature_maps = features

    var_maps = []
    for fea in feature_maps:
        fea = fea.permute(0, 2, 1)
        var = torch.var(fea, dim=1, keepdim=True)
        var = var.permute(0, 2, 1)
        var_maps.append(var)

    return var_maps


def token2feature(tokens, patch_grid_size):
    """add token transfer to feature"""
    B, L, D = tokens.shape
    # H = W = int(L**0.5)
    W, H = patch_grid_size[0], patch_grid_size[1]
    x = tokens.permute(0, 2, 1).view(B, D, H, W).contiguous()
    return x

In [29]:
encoder = "vitl"
max_depth = 1
size = (518, 518)
load_from = "/data/coding/code/da2-prompt-tuning/da2_metric_depth/exp/mvsec_voxel_nl_disp_da2vitl_20250107_152845/latest.pth"
outdir = "/data/coding/code/da2-prompt-tuning/vis_feature_maps/mvsec_nl_10_518"

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
# DEVICE = 'cpu'
model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

depth_anything = DepthAnythingV2(**{**model_configs[encoder], 'max_depth': max_depth})

checkpoint = torch.load(load_from, map_location='cpu')
if 'model' in checkpoint.keys():
    checkpoint = checkpoint['model']
    checkpoint = {
        (key[7:] if key.startswith("module.") else key): value
        for key, value in checkpoint.items()
    }
depth_anything.load_state_dict(checkpoint)
print(f"Model weights load from {load_from} successfully!")
depth_anything = depth_anything.to(DEVICE).eval()

In [31]:
from da2_metric_depth.dataset.eventscape import EventScape
from da2_metric_depth.dataset.mvsec import MVSEC
from torch.utils.data import DataLoader

# valset = EventScape(
#     "/home/sph/event/da2-prompt-tuning/dataset/splits/eventscape/test.txt",
#     "val",
#     normalized_d=False,
#     size=size,
# )

valset = MVSEC(
    "/data/coding/code/da2-prompt-tuning/dataset/splits/mvsec/vis_feature.txt",
    "val",
    normalized_d=False,
    size=size,
)

valloader = DataLoader(
        valset,
        batch_size=1,
        pin_memory=True,
        num_workers=4,
        drop_last=True,
    )

In [32]:
import os
import matplotlib
import numpy as np

os.makedirs(outdir, exist_ok=True)
npy_dir = os.path.join(outdir, 'npy')
vis_dir = os.path.join(outdir, 'vis')
fea_dir = os.path.join(outdir, 'fea')
os.makedirs(npy_dir, exist_ok=True)
os.makedirs(vis_dir, exist_ok=True)
os.makedirs(fea_dir, exist_ok=True)

cmap = matplotlib.colormaps.get_cmap('Spectral')

In [ ]:
depth_anything.eval()
for k, sample in enumerate(valloader):
    filename = sample["image_path"][0]
    print(f'Progress {k+1}/{len(valloader)}: {filename}')
    
    raw_image = cv2.imread(filename)
    h, w = raw_image.shape[0], raw_image.shape[1]
    
    with torch.no_grad():
        img = sample["image"].to(DEVICE)
        depth, var_maps = depth_anything(img)
        depth = F.interpolate(
            depth[:, None], (h, w), mode="bilinear", align_corners=True
        )[0, 0]
    
    depth = depth.cpu().numpy()

    # Save predicted depth map
    output_path = os.path.join(npy_dir, os.path.splitext(os.path.basename(filename))[0] + '.npy')
    np.save(output_path, depth)
    
    # vis depth_map
    depth = (depth - depth.min()) / (depth.max() - depth.min()) * 255.0
    depth = depth.astype(np.uint8)
    depth = (cmap(depth)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)
    
    output_path = os.path.join(vis_dir, os.path.splitext(os.path.basename(filename))[0] + '.png')
    # Merge depth map and raw rgb
    split_region = np.ones((raw_image.shape[0], 50, 3), dtype=np.uint8) * 255
    combined_result = cv2.hconcat([raw_image, split_region, depth])
    cv2.imwrite(output_path, combined_result)
    
    # vis feature map
    for i, var in enumerate(var_maps):
        output_path = os.path.join(fea_dir, os.path.splitext(os.path.basename(filename))[0] + f'_{i}.png')
        var = var.cpu().numpy().squeeze()
        # var = (cmap(var)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)
        # cv2.imwrite(output_path, var)
        np.save(output_path, var)

In [ ]:
# Single Image
import os
import matplotlib
import numpy as np

filename = "/data/coding/upload-data/data/DSEC/000767.png"
outdir = "/data/coding/upload-data/data/DSEC"
cmap = matplotlib.colormaps.get_cmap('Spectral')

raw_image = cv2.imread(filename)
depth, var_maps = depth_anything.infer_image(raw_image, 1036)

depth = (depth - depth.min()) / (depth.max() - depth.min()) * 255.0
depth = depth.astype(np.uint8)
depth = (cmap(depth)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)

cv2.imwrite(os.path.join(outdir, os.path.splitext(os.path.basename(filename))[0] + '_depth.png'), depth)
# vis feature map
for i, var in enumerate(var_maps):
    output_path = os.path.join(outdir, os.path.splitext(os.path.basename(filename))[0] + f'fea_{i}.npy')
    var = var.cpu().numpy().squeeze()
    # var = (cmap(var)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)
    # cv2.imwrite(output_path, var)
    np.save(output_path, var)

### epde

In [ ]:
class EPDEVisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        event_voxel_chans=5,
        embed_dim=768,
        depth=12,
        embed_layer=PatchEmbed,
        encoder="vitl",
        dataset="mvsec",
        max_depth=1,
        norm_layer=None,
        prompt_type=None,
        depth_anything_pretrained=None,
    ):
        super(EPDEVisionTransformer, self).__init__()

        self.encoder = encoder
        self.img_size = img_size
        self.patch_size = patch_size
        self.event_voxel_chans = event_voxel_chans
        self.embed_dim = embed_dim
        self.depth = depth
        self.prompt_depth = self.depth // 2
        self.max_depth = max_depth
        self.embed_layer = embed_layer
        self.norm_layer = norm_layer or partial(nn.LayerNorm, eps=1e-6)
        self.prompt_type = prompt_type
        self.depth_anything_pretrained = depth_anything_pretrained
        self.dataset = dataset

        depth_anything_model_configs = {
            "vits": {
                "encoder": "vits",
                "features": 64,
                "out_channels": [48, 96, 192, 384],
            },
            "vitb": {
                "encoder": "vitb",
                "features": 128,
                "out_channels": [96, 192, 384, 768],
            },
            "vitl": {
                "encoder": "vitl",
                "features": 256,
                "out_channels": [256, 512, 1024, 1024],
            },
            "vitg": {
                "encoder": "vitg",
                "features": 384,
                "out_channels": [1536, 1536, 1536, 1536],
            },
        }

        self.foundation = DepthAnythingV2(**depth_anything_model_configs[self.encoder])
        self.blocks_to_take = self.foundation.intermediate_layer_idx[encoder]
        self.num_heads = self.foundation.pretrained.num_heads

        self.patch_embed_prompt = embed_layer(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=event_voxel_chans,
            embed_dim=embed_dim,
        )

        prompt_blocks = []
        prompt_rectify = []
        prompt_fuse = []
        for i in range(self.prompt_depth):
            prompt_blocks.append(
                Block(
                    dim=embed_dim,
                    num_heads=self.num_heads,
                    norm_layer=self.norm_layer,
                    qkv_bias=True,
                )
            )
            prompt_rectify.append(MaxVar_Feat_Rect())

            if i in self.blocks_to_take:
                prompt_fuse.append(MaxVar_Feat_Fuse())
            else:
                prompt_fuse.append(nn.Identity())
        self.prompt_blocks = nn.Sequential(*prompt_blocks)
        self.prompt_rectify = nn.Sequential(*prompt_rectify)
        self.prompt_fuse = nn.Sequential(*prompt_fuse)

        self.init_weights()

    def init_weights(self):
        pass

    def _get_intermediate_layers_not_chunked(self, x, n=1):
        image = x[:, :3, :, :]
        event = x[:, 3:, :, :]
        B, nc, w, h = image.shape
        patch_grid_size = (w // self.patch_size, h // self.patch_size)

        # Compute event and image embedding
        image_token = self.foundation.pretrained.patch_embed(image)
        prompt_token = self.patch_embed_prompt(event)

        # Adding cls_token
        image_token = torch.cat(
            (
                self.foundation.pretrained.cls_token.expand(
                    image_token.shape[0], -1, -1
                ),
                image_token,
            ),
            dim=1,
        )
        prompt_token = torch.cat(
            (
                self.foundation.pretrained.cls_token.expand(
                    prompt_token.shape[0], -1, -1
                ),
                prompt_token,
            ),
            dim=1,
        )

        # Add positional encoding
        image_token += self.foundation.pretrained.interpolate_pos_encoding(
            image_token, w, h
        )
        prompt_token += self.foundation.pretrained.interpolate_pos_encoding(
            prompt_token, w, h
        )
        # prompt_token = self.prompt_blocks[0](prompt_token)

        image_fea, even_fea = []
        output, total_block_len = [], len(self.foundation.pretrained.blocks)
        blocks_to_take = (
            range(total_block_len - n, total_block_len) if isinstance(n, int) else n
        )

        for i, blk in enumerate(self.foundation.pretrained.blocks):
            if i < self.prompt_depth:
                # use [:, 1:] to exclude the cls_token
                image_feat = token2feature(image_token[:, 1:], patch_grid_size)
                prompt_feat = token2feature(prompt_token[:, 1:], patch_grid_size)

                image_feat, prompt_feat = self.prompt_rectify[i](
                    image_feat, prompt_feat
                )
                prompt_token = prompt_token.clone()
                # image_token = image_token.clone()
                prompt_token[:, 1:] = feature2token(prompt_feat)
                # image_token[:, 1:] = feature2token(image_feat)

                # prompt block
                prompt_token = self.prompt_blocks[i](prompt_token)

            image_token = blk(image_token)
            if i < self.prompt_depth and i in blocks_to_take:
                # Feature Fuse
                image_feat = token2feature(image_token[:, 1:], patch_grid_size)
                prompt_feat = token2feature(prompt_token[:, 1:], patch_grid_size)
                fuse_token = feature2token(self.prompt_fuse[i](image_feat, prompt_feat))
                cls_token = image_token[:, 0].unsqueeze(1)
                output.append(torch.cat((cls_token, fuse_token), dim=1))

            if i >= self.prompt_depth and i in blocks_to_take:
                output.append(image_token)

        assert len(output) == len(
            blocks_to_take
        ), f"only {len(output)} / {len(blocks_to_take)} blocks found"
        return output

    def get_intermediate_layers(
        self,
        x: torch.Tensor,
        n: Union[int, Sequence] = 1,  # Layers or n last layers to take
        return_class_token: bool = False,
        norm=True,
    ) -> Tuple[Union[torch.Tensor, Tuple[torch.Tensor]]]:
        outputs = self._get_intermediate_layers_not_chunked(x, n)

        if norm:
            outputs = [self.foundation.pretrained.norm(out) for out in outputs]

        class_tokens = [out[:, 0] for out in outputs]
        outputs = [out[:, 1:] for out in outputs]

        if return_class_token:
            return tuple(zip(outputs, class_tokens))
        return tuple(outputs)

    def forward(self, x):
        patch_h, patch_w = x.shape[-2] // 14, x.shape[-1] // 14
        features = self.get_intermediate_layers(
            x,
            self.foundation.intermediate_layer_idx[self.encoder],
            return_class_token=True,
        )

        depth = self.foundation.depth_head(features, patch_h, patch_w) * self.max_depth

        return depth.squeeze(1)

#### Vis feature Map

In [34]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
import os
# EventScape
# patch_size = (38, 19) # 266
# patch_size = (50, 25) # 350
# patch_size = (74, 37) # 518 
# patch_size = (148, 74) # 1036

# MVSEC
# patch_size = (25, 19) # 266
patch_size = (49, 37) # 518
# patch_size = (98, 74) # 1036

# DSEC
# patch_size = (99, 74)# 1036

In [27]:
def vis_depth_map(dep, show_colorbar=False, cmap_name="Spectral", save_fig=False, save_path="test.png"):
    cmap = matplotlib.colormaps.get_cmap(cmap_name)

    fig, ax = plt.subplots()
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.axis("off")

    im = ax.imshow(dep, cmap=cmap, extent=[0, dep.shape[1], 0, dep.shape[0]])
    if show_colorbar:
        plt.colorbar(im)

    dpi = fig.get_dpi()
    fig.set_size_inches(dep.shape[1] / dpi, dep.shape[0] / dpi)

    if save_fig:
        plt.savefig(save_path)
    plt.show()

In [ ]:
p = "/data/coding/code/da2-prompt-tuning/vis_feature_maps/mvsec_nl_10_518/npy/00010.npy"
d = np.load(p)
print(d.min(), d.max())
vis_depth_map(d)

In [ ]:
fea_path = "/data/coding/upload-data/data/DSEC/000040expofea_3.npy"
fea = torch.from_numpy(np.load(fea_path))
fea = fea.unsqueeze(0)
print(fea.shape)
fea = token2feature(fea, patch_size)
print(fea.shape)
var = torch.var(fea, dim=1, keepdim=True).squeeze().numpy()
print(var.shape)
print(var.min(), var.max())
vis_depth_map(var, cmap_name="viridis", save_fig=True)
fea = fea.numpy()

In [ ]:
dir = "/data/coding/code/da2-prompt-tuning/vis_feature_maps/mvsec_nl_10_518"
# patch_size = (25, 19) # 266
# patch_size = (49, 37) # 518
# patch_size = (98, 74) # 1036

fea_npy_dir = os.path.join(dir, "fea")
fea_vis_dir = os.path.join(dir, "fea_var_vis")
os.makedirs(fea_vis_dir, exist_ok=True)

fea_npy_paths = os.listdir(fea_npy_dir)
for p in fea_npy_paths:
    npy_path = os.path.join(fea_npy_dir, p)
    fea = torch.from_numpy(np.load(npy_path))
    fea = fea.unsqueeze(0)
    fea = token2feature(fea, patch_size)
    var = torch.var(fea, dim=1, keepdim=True).squeeze().numpy()
    
    file_name = p.split('_')[0] + '_' + p.split('_')[1][0] + ".png"
    out_path = os.path.join(fea_vis_dir, file_name)
    vis_depth_map(var, cmap_name="viridis", save_fig=True, save_path=out_path)

In [ ]:
dir = "/home/sph/event/da2-prompt-tuning/vis_feature_maps/da2_vitl_mvsec_fea_1036"

dep_npy_dir = os.path.join(dir, "npy")
dep_vis_dir = os.path.join(dir, "dep_vis")
os.makedirs(dep_vis_dir, exist_ok=True)

dep_npy_paths = os.listdir(dep_npy_dir)
for p in dep_npy_paths:
    npy_path = os.path.join(dep_npy_dir, p)
    dep = np.load(npy_path)
    
    file_name = p.replace("npy", "png")
    out_path = os.path.join(dep_vis_dir, file_name)
    vis_depth_map(dep, save_fig=True, save_path=out_path)